# Full Finetuning

## Prebuilt data form huggingface data hub

In [1]:
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, BitsAndBytesConfig

W0806 03:28:08.779000 2029844 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0806 03:28:08.795000 2029844 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
dataset = load_dataset("roneneldan/TinyStories", split="train")

In [3]:
print(dataset[0])

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [4]:
print(dataset[1])

{'text': 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were falling. Beep liked how the leaves fall and wanted to play with them. Beep drove under the tree and watched the leaves fall on him. He laughed and beeped his horn.\n\nBeep played with the falling leaves all day. When it was time to go home, Beep knew he needed more fuel. He went to the fuel place and got more healthy fuel. Now, Beep was ready to go fast and play again the next day. And Beep lived happily ever after.'}


## Our own custom data (non instruction data) for domain specific finetuning

In [5]:
import fitz # This is the library from PyMyPDF to load the pdf

### Adding the text from the pdf to a python list

In [6]:
def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [7]:
pdf_texts = extract_text_from_pdf("Metformin.pdf")

In [8]:
pdf_texts

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA red

### Chunking the data into paragraphs

In [9]:
import re
def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        # Split on double line breaks or long newlines
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30: # ignore too short lines
                paragraphs.append(clean)
    return paragraphs


In [10]:
paragraphs = split_paragraphs(pdf_texts)

In [11]:
paragraphs

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.',
 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA redu

### Creating a d

In [12]:
data = [{"text": p} for p in paragraphs]

In [13]:
data

[{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.'},
 {'text': 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits h

### Converting this data to huggingface compatible data

In [14]:
dataset = Dataset.from_list(data)

In [15]:
dataset

Dataset({
    features: ['text'],
    num_rows: 4
})

## Lets Select the Model

In [16]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

### Getting the tokenizer for this specific model

In [17]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [18]:
if tokenizer.pad_token is None: 
    tokenizer.pad_token = tokenizer.eos_token

### Data preprocessing = passing the text, trauncating (if the text is longer then 512 tokens, cut it off, keep the only first 512), padding (if the text is shorter than length of 512, pad it with <pad> tokens up to 512 length, max_length (fixed sequence length per training example))

In [19]:
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [20]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [21]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})

### Loading the model

In [22]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### Define training arguments

In [23]:
training_args = TrainingArguments(
    output_dir="./llama-pharma-domain",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    bf16=True,
    report_to="none"
)

In [24]:
help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing

### Setting up arguments for the trainer

In [25]:
trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=tokenized
)

In [26]:
import torch
print(torch.cuda.is_available())        # must be True
print(torch.cuda.get_device_name(0))    # should name your GB10

True
NVIDIA GB10


In [27]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4, training_loss=6.252408027648926, metrics={'train_runtime': 70.7764, 'train_samples_per_second': 0.113, 'train_steps_per_second': 0.057, 'total_flos': 25424176349184.0, 'train_loss': 6.252408027648926, 'epoch': 2.0})

# Now lets see the LORA based method

In [28]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [29]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [30]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [31]:
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="bfloat16")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [32]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [33]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [34]:
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"], 
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


In [35]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [36]:
tokenized

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})

### Setting up LoRA configurations

In [37]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [39]:
q_lora_model = get_peft_model(model, lora_config)

In [40]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=4,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    bf16=True,
    report_to="none"
)

In [41]:
trainer = Trainer(
    model=q_lora_model,
    args=args,
    train_dataset=tokenized
)

In [42]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=8, training_loss=9.781967163085938, metrics={'train_runtime': 1.7905, 'train_samples_per_second': 8.936, 'train_steps_per_second': 4.468, 'total_flos': 50903717511168.0, 'train_loss': 9.781967163085938, 'epoch': 4.0})

In [43]:
model_path = "/home/ankitanand/Documents/pp/Finetuning_HF/tinyllama-lora/checkpoint-8"

In [44]:
trained_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [45]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [48]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [49]:
outputs = trained_model.generate(
     **inputs, 
     max_new_tokens=100,
     temperature=0.8,
     top_p=0.9,
     do_sample=True,
     repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [50]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe was more effective than either drug alone in reducing the risk of heart attacks and strokes. Atorvastatin is not approved by the Food and Drug Administration (FDA) for use as a lipid lowering agent, but it is available under a specialty drug program for high cholesterol.
In an international clinical trial, researchers assessed the safety and efficacy of Ezetimibe with Atorvastatin compared to placebo or atorv
